In [ ]:
%pip install mlflow scikit-learn pandas -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import mlflow
import mlflow.sklearn

In [ ]:
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print(X.shape)
print(data.target_names)
X.head()

In [ ]:
print(X.describe())
print(y.value_counts())

In [ ]:
plt.figure(figsize=(6,4))
for target_value in y.unique():
    subset = X[y == target_value]
    plt.hist(subset["alcohol"], alpha=0.5, label=data.target_names[target_value])
plt.xlabel("Alcohol")
plt.ylabel("Frecuencia")
plt.legend()
plt.title("Distribución de 'alcohol' por clase")
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

In [ ]:
mlflow.set_experiment("clasificacion_vino")

n_estimators = 100
max_depth = 5

with mlflow.start_run(run_name="random_forest_wine"):

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro")
    rec = recall_score(y_test, y_pred, average="macro")
    f1 = f1_score(y_test, y_pred, average="macro")

    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(classification_report(y_test, y_pred, target_names=data.target_names))

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("model_type", "RandomForestClassifier")

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1_score", f1)

    mlflow.sklearn.log_model(model, "modelo_random_forest")

    print(mlflow.active_run().info.run_id)

In [ ]:
fastapi_code = """
from fastapi import FastAPI, Query
from transformers import pipeline

app = FastAPI(title="Mi API de Practica - FastAPI + Hugging Face")

sentiment_pipeline = pipeline("sentiment-analysis")
summarization_pipeline = pipeline("summarization")


@app.get("/")
def read_root():
    return {"mensaje": "Bienvenido a mi API de practica con FastAPI y Hugging Face"}


@app.get("/health")
def health_check():
    return {"status": "ok"}


@app.get("/sentiment")
def analyze_sentiment(text: str = Query(...)):
    result = sentiment_pipeline(text)
    return {"input": text, "result": result}


@app.get("/summarize")
def summarize_text(text: str = Query(...)):
    result = summarization_pipeline(text, max_length=60, min_length=10, do_sample=False)
    return {"input": text, "summary": result}


@app.get("/info")
def get_info():
    return {
        "nombre": "Practica FastAPI + HuggingFace",
        "endpoints": ["/", "/health", "/sentiment", "/summarize", "/info"]
    }
"""

print(fastapi_code)